# Live demo: your playlist, your face, your song

This notebook runs on **Google Colab**. It lets a visitor pick songs they like from a small library, takes a webcam photo, reads the facial expression with our classifier, and recommends a song that fits both the expression and the visitor's taste.

**Before the demo (once):** run section 1 and upload `demo_bundle.zip` (about 10 MB). It contains the trained model, the cleaned song catalogue and a few backup face images.

**During the demo:** run sections 2, 3 and 4 in order.

The webcam photo is kept in memory only: it is never written to disk and disappears when the notebook is closed. Ask the visitor for permission before taking it.

## 1. Setup (run before the demo)

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

if Path("demo/emotion_model.keras").exists():
    print("Demo files already present.")
else:
    uploaded = files.upload()                 # choose demo_bundle.zip
    with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
        z.extractall("demo")
    print("Demo files unpacked in demo/")

In [ ]:
import io
from base64 import b64decode
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DEMO_DIR = Path("demo")
EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

tracks = pd.read_csv(DEMO_DIR / "demo_tracks.csv")
library = pd.read_csv(DEMO_DIR / "library.csv")
print(f"Song catalogue: {len(tracks):,} well-known songs | library shown to the visitor: {len(library)} songs")

The functions below are the same as in the main notebook `mood_recommender.ipynb` (sections 4.1 to 4.4): emotion → point on the valence/energy map, driving rules, and the recommendation function.

In [ ]:
EMOTION_COORDS = {
    "happy":    (0.85, 0.70),
    "surprise": (0.65, 0.85),
    "angry":    (0.20, 0.90),
    "fear":     (0.15, 0.75),
    "disgust":  (0.20, 0.60),
    "sad":      (0.15, 0.25),
    "neutral":  (0.50, 0.45),
}
RISK_GROUPS = {
    "tension":       ["angry", "fear", "disgust"],
    "excitement":    ["happy", "surprise"],
    "low alertness": ["sad"],
    "none":          ["neutral"],
}
SAFE_POINTS = {"tension": (0.60, 0.35), "low alertness": (0.60, 0.60)}
TEMPO_RANGES = {"tension": (60, 100), "excitement": (60, 120), "low alertness": (85, 110), "none": (60, 120)}
STEER = 0.7
POPULARITY_LEVELS = [70, 50, 0]
MAX_DISTANCE = 0.15


def has_genre(genres_text, wanted):
    for genre in genres_text.split(";"):
        if genre in wanted:
            return True
    return False


def emotion_to_point(probabilities):
    valence, energy = 0.0, 0.0
    for emotion, probability in probabilities.items():
        emotion_valence, emotion_energy = EMOTION_COORDS[emotion]
        valence += probability * emotion_valence
        energy += probability * emotion_energy
    return valence, energy


def detect_driving_risk(probabilities):
    best_risk, best_score = "none", 0.0
    for risk, emotions in RISK_GROUPS.items():
        score = 0.0
        for emotion in emotions:
            score += probabilities[emotion]
        if score > best_score:
            best_risk, best_score = risk, score
    return best_risk


def driving_target(probabilities):
    valence, energy = emotion_to_point(probabilities)
    risk = detect_driving_risk(probabilities)
    if risk == "tension" or risk == "low alertness":
        safe_valence, safe_energy = SAFE_POINTS[risk]
        valence = valence + STEER * (safe_valence - valence)
        energy = energy + STEER * (safe_energy - energy)
    elif risk == "excitement":
        energy = min(energy, 0.60)
    return risk, (round(float(valence), 2), round(float(energy), 2)), TEMPO_RANGES[risk]


def recommend(probabilities, mode="match", taste=None, n=5, exclude=()):
    valence, energy = emotion_to_point(probabilities)
    candidates = tracks[~tracks["track_id"].isin(exclude)]

    if mode == "drive":
        risk, (valence, energy), (tempo_min, tempo_max) = driving_target(probabilities)
        candidates = candidates[candidates["tempo"].between(tempo_min, tempo_max)]

    if taste is not None and len(taste["genres"]) > 0:
        in_taste = candidates["genres"].apply(has_genre, wanted=taste["genres"])
        if candidates.loc[in_taste, "artists"].nunique() >= n:
            candidates = candidates[in_taste]

    distance = np.sqrt((candidates["valence"] - valence) ** 2 + (candidates["energy"] - energy) ** 2)
    candidates = candidates.assign(distance=distance.round(3))

    for level in POPULARITY_LEVELS:
        close = candidates[(candidates["popularity"] >= level) & (candidates["distance"] <= MAX_DISTANCE)]
        if close["artists"].nunique() >= n:
            break
    if close["artists"].nunique() < n:
        close = candidates

    best = close.sort_values("distance").drop_duplicates("artists").head(n)
    return best[["track_id", "track_name", "artists", "genres", "valence", "energy", "tempo", "popularity", "distance"]]


def taste_from_songs(songs):
    """Taste profile from a table of liked songs: the genres cited at least half as often as the top genre."""
    genre_counts = {}
    for genres_text in songs["genres"]:
        for genre in genres_text.split(";"):
            genre_counts[genre] = genre_counts.get(genre, 0) + 1
    genres = set()
    if len(genre_counts) > 0:
        top_count = max(genre_counts.values())
        for genre, count in genre_counts.items():
            if count >= top_count / 2:
                genres.add(genre)
    return {"genres": genres}


print("Functions ready.")

In [ ]:
import cv2
from tensorflow import keras

model = keras.models.load_model(DEMO_DIR / "emotion_model.keras")


def predict_emotions(face48):
    """face48: a 48x48 grayscale PIL image. Returns {emotion: probability}.
    The image and its mirror are both predicted and averaged, which makes the result a little more stable."""
    x = np.array(face48.convert("L"), dtype="float32")[None, ..., None] / 255.0
    proba = (model.predict(x, verbose=0)[0] + model.predict(x[:, :, ::-1, :], verbose=0)[0]) / 2
    return {emotion: float(p) for emotion, p in zip(EMOTIONS, proba)}


# Self-check: the model must recognise most of the backup faces whose true emotion we know
backup_paths = sorted((DEMO_DIR / "sample_faces").glob("test/*/*.jpg"))
correct = 0
for path in backup_paths:
    probabilities = predict_emotions(Image.open(path))
    if max(probabilities, key=probabilities.get) == path.parent.name:
        correct += 1
print(f"Model loaded. Self-check on backup faces: {correct}/{len(backup_paths)} correct "
      f"(around half or more is expected; 0 to 3 would mean a loading problem).")

In [ ]:
# OpenCV's standard face detector. If it is not available in this Colab version, we fall back to the centre of the photo.
try:
    face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    DETECTOR_OK = not face_detector.empty()
except Exception:
    DETECTOR_OK = False
print("Face detector available:", DETECTOR_OK)
if not DETECTOR_OK:
    print("-> The centre of the photo will be used: ask the visitor to put their face in the middle of the image.")


def extract_face(photo):
    """Find the largest face in a PIL photo and return it as a 48x48 grayscale image, like the training images.
    If no face is found, the central part of the photo is used instead."""
    gray = np.array(photo.convert("L"))
    boxes = []
    if DETECTOR_OK:
        boxes = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(boxes) > 0:
        x, y, w, h = max(boxes, key=lambda box: box[2] * box[3])
        found = True
    else:
        height, width = gray.shape
        side = int(min(height, width) * 0.6)
        x, y, w, h = (width - side) // 2, (height - side) // 2, side, side
        found = False
    face = Image.fromarray(gray[y:y + h, x:x + w]).resize((48, 48))
    return face, found

In [ ]:
from IPython.display import HTML, IFrame, Javascript, display
from google.colab.output import eval_js


def take_photo(quality=0.9):
    """Show the webcam with a 'Capture' button and return the photo as a PIL image (kept in memory only)."""
    display(Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const capture = document.createElement('button');
          capture.textContent = 'Capture';
          capture.style.fontSize = '18px';
          capture.style.margin = '8px 0';
          div.appendChild(capture);
          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => capture.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
    '''))
    data = eval_js("takePhoto({})".format(quality))
    return Image.open(io.BytesIO(b64decode(data.split(",")[1])))


def show_song(song, label):
    """Print one recommended song and show a Spotify player for it."""
    print(f"{label}: {song['track_name']} - {song['artists']}  "
          f"(valence {song['valence']:.2f}, energy {song['energy']:.2f}, tempo {song['tempo']:.0f} BPM)")
    display(IFrame(f"https://open.spotify.com/embed/track/{song['track_id']}", width=480, height=152))

In [ ]:
def run_demo(face48, source_image=None, face_found=True):
    """Read the emotions of a 48x48 face, show them, recommend songs and add them to the playlist."""
    global playlist
    probabilities = predict_emotions(face48)
    valence, energy = emotion_to_point(probabilities)
    risk, drive_point, tempo_range = driving_target(probabilities)

    # What the classifier sees
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1.2, 0.8, 2]})
    if source_image is not None:
        axes[0].imshow(source_image)
        axes[0].set_title("Photo")
    axes[0].axis("off")
    axes[1].imshow(face48, cmap="gray")
    axes[1].set_title("What the model sees (48x48)" if face_found else "No face found: centre of the photo")
    axes[1].axis("off")
    order = sorted(EMOTIONS, key=lambda emotion: probabilities[emotion])
    axes[2].barh(order, [probabilities[e] * 100 for e in order], color="#4C72B0")
    for i, emotion in enumerate(order):
        axes[2].text(probabilities[emotion] * 100 + 1, i, f"{probabilities[emotion]:.0%}", va="center")
    axes[2].set_xlim(0, 110)
    axes[2].set_xlabel("Probability (%)")
    axes[2].set_title("Detected emotions")
    plt.tight_layout()
    plt.show()

    print(f"Point on the map: valence {valence:.2f}, energy {energy:.2f}")
    print(f"Driving risk: {risk} -> drive-mode target: valence {drive_point[0]}, energy {drive_point[1]}, "
          f"tempo {tempo_range[0]}-{tempo_range[1]} BPM")
    print(f"Taste used: {sorted(taste['genres'])}\n")

    already = list(playlist["track_id"])
    match_song = recommend(probabilities, mode="match", taste=taste, n=1, exclude=already).iloc[0]
    show_song(match_song, "MATCH MODE (fits your mood)")
    drive_song = recommend(probabilities, mode="drive", taste=taste, n=1,
                           exclude=already + [match_song["track_id"]]).iloc[0]
    show_song(drive_song, "DRIVE MODE (keeps you safe)")

    # The drive-mode song joins the playlist
    new_row = drive_song[["track_id", "track_name", "artists", "genres", "valence", "energy", "tempo"]].to_frame().T
    new_row["source"] = f"recommended ({risk})"
    playlist = pd.concat([playlist, new_row], ignore_index=True)
    print("\nPlaylist after this recommendation:")
    display(playlist[["track_name", "artists", "source"]])

## 2. The library: tick the songs you like

Tick a few songs (three or more works best), then run section 3.

In [ ]:
import ipywidgets as widgets

checkboxes = []
blocks = []
for family in library["family"].unique():
    boxes = []
    for _, song in library[library["family"] == family].iterrows():
        box = widgets.Checkbox(value=False, description=f"{song['track_name']} - {song['artists']}"[:60],
                               indent=False, layout=widgets.Layout(width="auto"))
        box.track_id = song["track_id"]
        boxes.append(box)
        checkboxes.append(box)
    blocks.append(widgets.VBox([widgets.HTML(f"<b>{family}</b>")] + boxes))

display(widgets.GridBox(blocks, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", grid_gap="12px")))

## 3. Your playlist and your taste

In [ ]:
chosen_ids = [box.track_id for box in checkboxes if box.value]
if len(chosen_ids) == 0:
    print("No song ticked: tick a few songs in section 2 and run this cell again.")
    print("Until then the recommendation will not use any taste.")

playlist = tracks[tracks["track_id"].isin(chosen_ids)][
    ["track_id", "track_name", "artists", "genres", "valence", "energy", "tempo"]].copy()
playlist["source"] = "chosen by the visitor"
playlist = playlist.reset_index(drop=True)
taste = taste_from_songs(playlist)

print(f"{len(playlist)} songs in the playlist | taste genres: {sorted(taste['genres'])}")
if len(playlist) > 0:
    print(f"Usual valence {playlist['valence'].mean():.2f} | usual energy {playlist['energy'].mean():.2f} "
          f"| usual tempo {playlist['tempo'].mean():.0f} BPM")
playlist[["track_name", "artists", "genres"]]

## 4. Your face → your song

Run the cell, allow the camera in the browser, make an expression and click **Capture**. A clear smile or a surprised face works best. Run the cell again for another photo: each run adds one song to the playlist.

In [ ]:
photo = take_photo()
face, found = extract_face(photo)
run_demo(face, source_image=photo, face_found=found)
del photo                      # the photo is not kept

### No webcam? Upload a photo instead

Take a selfie with a phone, send it to this computer, run the cell and choose the photo (several photos can be chosen at once). The photos are kept in memory only.

In [ ]:
from google.colab import files
from PIL import ImageOps

uploaded = files.upload()                                  # choose one or several photos
for name, content in uploaded.items():
    print("=====", name)
    photo = ImageOps.exif_transpose(Image.open(io.BytesIO(content))).convert("RGB")   # upright, in colour
    face, found = extract_face(photo)
    run_demo(face, source_image=photo, face_found=found)
del photo, uploaded                                        # the photos are not kept

### Backup: use a test face instead of the webcam

If the camera is blocked or the room is too dark, choose one of the stored FER-2013 test faces. Change the emotion name to try another one.

In [ ]:
BACKUP_EMOTION = "angry"       # angry, disgust, fear, happy, neutral, sad or surprise
BACKUP_NUMBER = 0              # 0, 1 or 2

backup_path = sorted((DEMO_DIR / "sample_faces" / "test" / BACKUP_EMOTION).glob("*.jpg"))[BACKUP_NUMBER]
backup_face = Image.open(backup_path)
run_demo(backup_face, source_image=backup_face.convert("RGB"))